# Step 3 - GRN optimization

Per-gene parameter fitting per condition, then joint signed edge deletion
across both networks. The paper uses `n = 0.5, k = 1.5`.

In [ ]:
import os
import sys

NETDESDUO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "neutrophil_data"))
sys.path.insert(0, NETDESDUO_ROOT)
import NetDesDuo
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import random
import importlib
import math
import joblib
from sklearn.metrics import mean_squared_error
importlib.reload(NetDesDuo)
import math
import matplotlib.pyplot as plt

# step-3 run tree - not committed, edit for your setup
WORK_DIR = "/projects/lulab/alex/neutrophil_case/network_optimization"
os.chdir(WORK_DIR)

## Naive: expression, initial network, per-gene fitting

In [ ]:
tf11=pd.read_csv('naive_results/naive_expression_data.csv')
genes = pd.read_csv('naive_results/naive_genes_expressed.csv')
genes = genes['x'].to_list()
tf11 = tf11.transpose()
tf11 = tf11[1:100000]
tf11.index = genes

In [ ]:
#restore the average pseudotime values to before log-ing
tf12expression = tf11.apply(lambda x: (math.e**x - 1))
tf12expression[tf12expression <= 0.001] = 0.001
#scale expression
tf12_scaled = tf12expression.div(tf12expression.max(axis=1),axis=0)
tf12expression_logtarget = tf12_scaled.apply(lambda x: np.log2(x))
tf12expression_logtarget = tf12expression_logtarget.dropna()
tf12expression = tf12expression.dropna()

In [ ]:
tf12expression_logtarget.to_csv("naive_results/naive_log_expression.csv", index = True)
tf12expression.to_csv("naive_results/naive_scaled_expression.csv", index = True)

In [ ]:
network0 = pd.read_csv('naive_results/naive_initial_network.csv')
network1 = network0[["Source", "Target", "Interaction"]]
gene_list = list(pd.unique(network1['Target']))
network1.columns = ['Source', 'Target', 'Interaction']
ids = list(range(len(gene_list)))

In [ ]:
results = joblib.Parallel(n_jobs=15, backend="loky")(
    joblib.delayed(NetDesDuo.MSE_stPoint_one_gene)(gene_list, network1, tf12expression, tf12expression_logtarget,
                                               ii) for ii in range(len(gene_list)))
all_st = [st for (st, df) in results]
MSE_st = []
for st, df in results:
    df = df.copy()
    df["SP"] = st
    MSE_st.append(df)
Part1 = [all_st, MSE_st]
joblib.dump(Part1, "naive_results/naive_Part1.joblib")

In [ ]:
Part1 = joblib.load("naive_results/naive_Part1.joblib")
results = joblib.Parallel(n_jobs=15, backend="loky")(
    joblib.delayed(NetDesDuo.MSE_top_one_gene)(
        gene_list, network1, tf12expression, tf12expression_logtarget, Part1, ii
    )
    for ii in range(len(gene_list))
)

selected_total = [sel for (sel, st, rp, mse_min, df) in results]
all_st2        = [st  for (sel, st, rp, mse_min, df) in results]
res_par        = [rp  for (sel, st, rp, mse_min, df) in results]
MSE_min_total  = [mse_min for (sel, st, rp, mse_min, df) in results]

MSE_st2 = []
for sel, st, rp, mse_min, df in results:
    df = df.copy()
    df['SP'] = st
    MSE_st2.append(df)

Part2 = [selected_total, all_st2, res_par, MSE_min_total, MSE_st2]
joblib.dump(Part2, "naive_results/naive_Part2.joblib")

In [ ]:
w_tot=NetDesDuo.tuning(MSE_table2=Part2,nums=20,gene_list=gene_list)
Part3=NetDesDuo.MSE_bin(MSE_table2=Part2,w_tot=w_tot,gene_list=gene_list, 
                        network=network1, expression=tf12expression,log_expression=tf12expression_logtarget)
consis_results=NetDesDuo.consis_test(gene_list=gene_list,MSE_table=Part3)
print("Part3")
joblib.dump(Part3, "naive_results/naive_Part3.joblib")
joblib.dump(consis_results, "naive_results/naive_consis_res.joblib")

In [ ]:
Part3 = joblib.load("naive_results/naive_Part3.joblib")
consis_results = joblib.load("naive_results/naive_consis_res.joblib")
int_test=NetDesDuo.interactions_test(gene_list=gene_list,network=network1, consis=consis_results,MSE_table=Part3,
                                    expression=tf12expression,log_expression=tf12expression_logtarget)
print("Interactions Test")
#MSE_cut here doesnt matter
delete = NetDesDuo.delete_int(gene_list = gene_list,network = network1,int_results = int_test,
                            consis_res = consis_results, MSE_cut = 100)
    
joblib.dump(delete, "naive_results/naive_delete.joblib")
joblib.dump(int_test, "naive_results/naive_int_test.joblib")

## Tumour-bearing: expression, initial network, per-gene fitting

In [ ]:
#tf11=pd.read_csv('/Users/alexren/desktop/Neutrophil/cancer_expression_data.csv')
tf11=pd.read_csv('cancer_results/cancer_expression_data.csv')
#genes = pd.read_csv('/Users/alexren/desktop/Neutrophil/cancer_genes_expressed.csv')
genes = pd.read_csv('cancer_results/cancer_genes_expressed.csv')
genes = genes['x'].to_list()
tf11 = tf11.transpose()
tf11 = tf11[1:100000]
tf11.index = genes

In [ ]:
#restore the average pseudotime values to before log-ing
tf12expression = tf11.apply(lambda x: (math.e**x - 1))
tf12expression[tf12expression <= 0.001] = 0.001
#scale expression
tf12_scaled = tf12expression.div(tf12expression.max(axis=1),axis=0)
tf12expression_logtarget = tf12_scaled.apply(lambda x: np.log2(x))
tf12expression_logtarget = tf12expression_logtarget.dropna()
tf12expression = tf12expression.dropna()

In [ ]:
tf12expression_logtarget.to_csv("cancer_results/cancer_log_expression.csv", index = True)

In [ ]:
network0 = pd.read_csv('cancer_results/cancer_initial_network.csv')
network1 = network0[["Source", "Target", "Interaction"]]
gene_list = list(pd.unique(network1['Target']))
network1.columns = ['Source', 'Target', 'Interaction']
ids = list(range(len(gene_list)))

In [ ]:
results = joblib.Parallel(n_jobs=15, backend="loky")(
    joblib.delayed(NetDesDuo.MSE_stPoint_one_gene)(gene_list, network1, tf12expression, tf12expression_logtarget,
                                               ii) for ii in range(len(gene_list)))
all_st = [st for (st, df) in results]
MSE_st = []
for st, df in results:
    df = df.copy()
    df["SP"] = st
    MSE_st.append(df)
Part1 = [all_st, MSE_st]
joblib.dump(Part1, "cancer_results/cancer_Part1.joblib")

In [ ]:
Part1 = joblib.load("cancer_results/cancer_Part1.joblib")
results = joblib.Parallel(n_jobs=5, backend="loky")(
    joblib.delayed(NetDesDuo.MSE_top_one_gene)(
        gene_list, network1, tf12expression, tf12expression_logtarget, Part1, ii
    )
    for ii in range(len(gene_list))
)

selected_total = [sel for (sel, st, rp, mse_min, df) in results]
all_st2        = [st  for (sel, st, rp, mse_min, df) in results]
res_par        = [rp  for (sel, st, rp, mse_min, df) in results]
MSE_min_total  = [mse_min for (sel, st, rp, mse_min, df) in results]

MSE_st2 = []
for sel, st, rp, mse_min, df in results:
    df = df.copy()
    df['SP'] = st
    MSE_st2.append(df)

Part2 = [selected_total, all_st2, res_par, MSE_min_total, MSE_st2]
joblib.dump(Part2, "cancer_results/cancer_Part2.joblib")

In [ ]:
w_tot=NetDesDuo.tuning(MSE_table2=Part2,nums=20,gene_list=gene_list)
Part3=NetDesDuo.MSE_bin(MSE_table2=Part2,w_tot=w_tot,gene_list=gene_list, 
                        network=network1, expression=tf12expression,log_expression=tf12expression_logtarget)
consis_results=NetDesDuo.consis_test(gene_list=gene_list,MSE_table=Part3)
print("Part3")
joblib.dump(Part3, "cancer_results/cancer_Part3.joblib")
joblib.dump(consis_results, "cancer_results/cancer_consis_res.joblib")

In [ ]:
Part3 = joblib.load("cancer_results/cancer_Part3.joblib")
consis_results = joblib.load("cancer_results/cancer_consis_res.joblib")
int_test=NetDesDuo.interactions_test(gene_list=gene_list,network=network1, consis=consis_results,MSE_table=Part3,
                                    expression=tf12expression,log_expression=tf12expression_logtarget)
print("Interactions Test")
#MSE_cut here doesnt matter
delete = NetDesDuo.delete_int(gene_list = gene_list,network = network1,int_results = int_test,
                            consis_res = consis_results, MSE_cut = 100)
    
joblib.dump(delete, "cancer_results/cancer_delete.joblib")
joblib.dump(int_test, "cancer_results/cancer_int_test.joblib")

## Joint signed edge deletion

In [ ]:
network = pd.read_csv('naive_results/naive_initial_network.csv')
network_naive = network[["Source", "Target", "Interaction"]]
genes_naive = list(pd.unique(network_naive['Target']))
tf12expression_naive = pd.read_csv("naive_results/naive_scaled_expression.csv", index_col = 0)
tf12expression_logtarget_naive = pd.read_csv("naive_results/naive_log_expression.csv", index_col = 0)
network_naive.columns = ['Source', 'Target', 'Interaction']
delete_naive = joblib.load("naive_results/naive_delete.joblib")
consis_res_naive = joblib.load("naive_results/naive_consis_res.joblib")
int_test_naive = joblib.load("naive_results/naive_int_test.joblib")
Part3_naive = joblib.load("naive_results/naive_Part3.joblib")

network_cancer = network1
genes_cancer = gene_list
tf12expression_cancer = tf12expression
tf12expression_logtarget_cancer = tf12expression_logtarget
delete_cancer = joblib.load("cancer_results/cancer_delete.joblib")
consis_res_cancer = joblib.load("cancer_results/cancer_consis_res.joblib")
int_test_cancer = joblib.load("cancer_results/cancer_int_test.joblib")
Part3_cancer = joblib.load("cancer_results/cancer_Part3.joblib")

In [ ]:
def process_full_pipeline(k, n):
    naive_dir  = f"naive_results/naive_networks/n_{n}_k_{k}"
    cancer_dir = f"cancer_results/cancer_networks/n_{n}_k_{k}"
    os.makedirs(naive_dir,  exist_ok=True)
    os.makedirs(cancer_dir, exist_ok=True)

    #deletion
    ob_newall_naive_s, ob_newall_cancer_s, delt_value_naive_s, delt_value_cancer_s, \
    st_new_cut_naive_s, st_new_cut_cancer_s = NetDesDuo.run_delete_combined_signed(
        network_naive, genes_naive, delete_naive, consis_res_naive, int_test_naive, Part3_naive,
        network_cancer, genes_cancer, delete_cancer, consis_res_cancer, int_test_cancer, Part3_cancer,
        n, k)

    naive_optimized_s  = NetDesDuo.ob_to_network(genes_naive,  ob_newall_naive_s)
    cancer_optimized_s = NetDesDuo.ob_to_network(genes_cancer, ob_newall_cancer_s)

    naive_optimized_s.to_csv(f"{naive_dir}/naive_optimized_network.csv",    index=False)
    cancer_optimized_s.to_csv(f"{cancer_dir}/cancer_optimized_network.csv", index=False)

    del_final_naive_s  = [delt_value_naive_s,  st_new_cut_naive_s,  ob_newall_naive_s]
    del_final_cancer_s = [delt_value_cancer_s, st_new_cut_cancer_s, ob_newall_cancer_s]

    joblib.dump(del_final_naive_s,  f"{naive_dir}/naive_delete_final_signed.joblib")
    joblib.dump(del_final_cancer_s, f"{cancer_dir}/cancer_delete_final_signed.joblib")

    #naive: parameter fitting to get final MSE
    Part4_naive = NetDesDuo.MSE_delete_fitting(
        gene_list      = genes_naive,
        network        = network_naive,
        expression     = tf12expression_naive,
        log_expression = tf12expression_logtarget_naive,
        del_int        = del_final_naive_s)

    joblib.dump(Part4_naive, f"{naive_dir}/naive_Part4_signed.joblib")

    w_tot_naive = NetDesDuo.tuning(
        MSE_table2 = Part4_naive,
        nums       = 20,
        gene_list  = genes_naive)

    res_final_naive = NetDesDuo.parameter_fitting(
        MSE_del_table  = Part4_naive,
        w_tot          = w_tot_naive,
        gene_list      = genes_naive,
        del_int        = del_final_naive_s,
        network        = network_naive,
        expression     = tf12expression_naive,
        log_expression = tf12expression_logtarget_naive)

    ob_newall_naive  = Part4_naive[1].copy()
    res_final_naive  = res_final_naive[0]

    joblib.dump(res_final_naive, f"{naive_dir}/naive_res_final_signed.joblib")
    joblib.dump(ob_newall_naive, f"{naive_dir}/naive_ob_newall_signed.joblib")

    #cancer: parameter fitting
    Part4_cancer = NetDesDuo.MSE_delete_fitting(
        gene_list      = genes_cancer,
        network        = network_cancer,
        expression     = tf12expression_cancer,
        log_expression = tf12expression_logtarget_cancer,
        del_int        = del_final_cancer_s)

    joblib.dump(Part4_cancer, f"{cancer_dir}/cancer_Part4_signed.joblib")

    w_tot_cancer = NetDesDuo.tuning(
        MSE_table2 = Part4_cancer,
        nums       = 20,
        gene_list  = genes_cancer)

    res_final_cancer = NetDesDuo.parameter_fitting(
        MSE_del_table  = Part4_cancer,
        w_tot          = w_tot_cancer,
        gene_list      = genes_cancer,
        del_int        = del_final_cancer_s,
        network        = network_cancer,
        expression     = tf12expression_cancer,
        log_expression = tf12expression_logtarget_cancer)

    ob_newall_cancer  = Part4_cancer[1].copy()
    res_final_cancer  = res_final_cancer[0]

    joblib.dump(res_final_cancer, f"{cancer_dir}/cancer_res_final_signed.joblib")
    joblib.dump(ob_newall_cancer, f"{cancer_dir}/cancer_ob_newall_signed.joblib")

In [ ]:
k_fixed = 1.0
n_vals  = [i / 10 for i in range(20)]

results = joblib.Parallel(n_jobs=20, backend="loky")(
    joblib.delayed(process_full_pipeline)(k_fixed, n) for n in n_vals
)

In [ ]:
n_fixed = 0.5
k_vals  = [i / 10 for i in range(30)]

results = joblib.Parallel(n_jobs=10, backend="loky")(
    joblib.delayed(process_full_pipeline)(k, n_fixed) for k in k_vals
)

## Sweep summary (figure S4)

In [ ]:
k_fixed = 1.0
n_vals = [i/10 for i in range(20)]

naive_sizes = []
cancer_sizes = []

for n in n_vals:
    naive_opt = pd.read_csv(f"naive_results/naive_networks/n_{n}_k_{k_fixed}/naive_optimized_network.csv")
    cancer_opt = pd.read_csv(f"cancer_results/cancer_networks/n_{n}_k_{k_fixed}/cancer_optimized_network.csv")
    
    
    naive_sizes.append(len(naive_opt))
    cancer_sizes.append(len(cancer_opt))

In [ ]:
k_fixed = 1.0
n_vals  = [i / 10 for i in range(20)]

naive_mse_vals  = []
cancer_mse_vals = []

for n in n_vals:
    naive_dir  = f"naive_results/naive_networks/n_{n}_k_{k_fixed}"
    cancer_dir = f"cancer_results/cancer_networks/n_{n}_k_{k_fixed}"

    # --- Load naive results ---
    ob_newall_naive  = joblib.load(f"{naive_dir}/naive_ob_newall_signed.joblib")
    res_final_naive  = joblib.load(f"{naive_dir}/naive_res_final_signed.joblib")

    # --- Load cancer results ---
    ob_newall_cancer = joblib.load(f"{cancer_dir}/cancer_ob_newall_signed.joblib")
    res_final_cancer = joblib.load(f"{cancer_dir}/cancer_res_final_signed.joblib")

    # --- Compute mean MSE for naive and cancer ---
    _, mean_mse_naive = NetDesDuo.calculate_network_mse(
        gene_list          = genes_naive,
        network            = network_naive,
        tfexpression       = tf12expression_naive,
        ob_newall          = ob_newall_naive,
        res_final          = res_final_naive,
        tfexpression_logtarget = tf12expression_logtarget_naive)

    _, mean_mse_cancer = NetDesDuo.calculate_network_mse(
        gene_list          = genes_cancer,
        network            = network_cancer,
        tfexpression       = tf12expression_cancer,
        ob_newall          = ob_newall_cancer,
        res_final          = res_final_cancer,
        tfexpression_logtarget = tf12expression_logtarget_cancer)

    naive_mse_vals.append(mean_mse_naive)
    cancer_mse_vals.append(mean_mse_cancer)

In [ ]:
jacc_vals = []
n_fixed = 0.5
k_vals = [i/10 for i in range(30)]
#temporary, remove later
k_vals.remove(2)

for k in k_vals:
    naive_opt = pd.read_csv(f"naive_results/naive_networks/n_{n_fixed}_k_{k}/naive_optimized_network.csv")
    cancer_opt = pd.read_csv(f"cancer_results/cancer_networks/n_{n_fixed}_k_{k}/cancer_optimized_network.csv")
    
    shared_nodes = [gene for gene in genes_cancer if gene in genes_naive]

    naive_sub = naive_opt[naive_opt["Source"].isin(shared_nodes) & naive_opt["Target"].isin(shared_nodes)]
    cancer_sub = cancer_opt[cancer_opt["Source"].isin(shared_nodes) & cancer_opt["Target"].isin(shared_nodes)]

    naive_edges  = set(zip(naive_sub["Source"], naive_sub["Target"]))
    cancer_edges = set(zip(cancer_sub["Source"], cancer_sub["Target"]))

    jacc = len(naive_edges.intersection(cancer_edges)) / len(naive_edges.union(cancer_edges))
    jacc_vals.append(jacc)

In [ ]:
n_fixed = 0.5
k_vals  = [i / 10 for i in range(30)]
#temporary, run this later
k_vals.remove(2)
naive_mse_vals  = []
cancer_mse_vals = []

for k in k_vals:
    naive_dir  = f"naive_results/naive_networks/n_{n_fixed}_k_{k}"
    cancer_dir = f"cancer_results/cancer_networks/n_{n_fixed}_k_{k}"

    # --- Load naive results ---
    ob_newall_naive  = joblib.load(f"{naive_dir}/naive_ob_newall_signed.joblib")
    res_final_naive  = joblib.load(f"{naive_dir}/naive_res_final_signed.joblib")

    # --- Load cancer results ---
    ob_newall_cancer = joblib.load(f"{cancer_dir}/cancer_ob_newall_signed.joblib")
    res_final_cancer = joblib.load(f"{cancer_dir}/cancer_res_final_signed.joblib")

    # --- Compute mean MSE for naive and cancer ---
    _, mean_mse_naive = NetDesDuo.calculate_network_mse(
        gene_list          = genes_naive,
        network            = network_naive,
        tfexpression       = tf12expression_naive,
        ob_newall          = ob_newall_naive,
        res_final          = res_final_naive,
        tfexpression_logtarget = tf12expression_logtarget_naive)

    _, mean_mse_cancer = NetDesDuo.calculate_network_mse(
        gene_list          = genes_cancer,
        network            = network_cancer,
        tfexpression       = tf12expression_cancer,
        ob_newall          = ob_newall_cancer,
        res_final          = res_final_cancer,
        tfexpression_logtarget = tf12expression_logtarget_cancer)

    naive_mse_vals.append(mean_mse_naive)
    cancer_mse_vals.append(mean_mse_cancer)

## Final networks at n = 0.5, k = 1.5

In [ ]:
#final network evaluation with signed method
#from plots choose n = 0.5 and k = 1.5 
ob_newall_naive_s, ob_newall_cancer_s, delt_value_naive_s, delt_value_cancer_s, st_new_cut_naive_s, st_new_cut_cancer_s = NetDesDuo.run_delete_combined_signed(
    network_naive, genes_naive, delete_naive, consis_res_naive, int_test_naive, Part3_naive, 
    network_cancer, genes_cancer, delete_cancer, consis_res_cancer, int_test_cancer, Part3_cancer, 
                                                          0.5, 1.5)
naive_optimized_s = NetDesDuo.ob_to_network(genes_naive, ob_newall_naive_s)
cancer_optimized_s = NetDesDuo.ob_to_network(genes_cancer, ob_newall_cancer_s)

naive_optimized_s.to_csv("naive_results/naive_optimized_network_signed.csv", index = False)
cancer_optimized_s.to_csv("cancer_results/cancer_optimized_network_signed.csv", index = False)

del_final_naive_s = [delt_value_naive_s, st_new_cut_naive_s, ob_newall_naive_s]
del_final_cancer_s = [delt_value_cancer_s, st_new_cut_cancer_s, ob_newall_cancer_s]

joblib.dump(del_final_naive_s, "naive_results/naive_delete_final_signed.joblib")
joblib.dump(del_final_cancer_s, "cancer_results/cancer_delete_final_signed.joblib")

## Naive exports

In [ ]:
tf11=pd.read_csv('naive_results/naive_expression_data.csv')
genes = pd.read_csv('naive_results/naive_genes_expressed.csv')
genes = genes['x'].to_list()
tf11 = tf11.transpose()
tf11 = tf11[1:100000]
tf11.index = genes

In [ ]:
#restore the average pseudotime values to before log-ing
tf12expression = tf11.apply(lambda x: (math.e**x - 1))
tf12expression[tf12expression <= 0.001] = 0.001
#scale expression
tf12_scaled = tf12expression.div(tf12expression.max(axis=1),axis=0)
tf12expression_logtarget = tf12_scaled.apply(lambda x: np.log2(x))
tf12expression_logtarget = tf12expression_logtarget.dropna()
tf12expression = tf12expression.dropna()

In [ ]:
network0 = pd.read_csv('naive_results/naive_initial_network.csv')
network1 = network0[["Source", "Target", "Interaction"]]
gene_list = list(pd.unique(network1['Target']))
network1.columns = ['Source', 'Target', 'Interaction']
ids = list(range(len(gene_list)))

In [ ]:
ob_newall = joblib.load("naive_results/naive_networks/n_0.5_k_1.5/naive_ob_newall_signed.joblib")
res_final = joblib.load("naive_results/naive_networks/n_0.5_k_1.5/naive_res_final_signed.joblib")
combined_network = NetDesDuo.save_combined_network(gene_list, ob_newall, res_final, output="naive_results/naive_combined_signed.csv")

export_network = pd.read_csv("naive_results/naive_combined_signed.csv")[["Source", "Target", "Interaction"]]
for j in range(len(export_network["Interaction"])):
    if export_network["Interaction"][j] > 1:
        export_network["Interaction"][j] = 1
    else:
        export_network["Interaction"][j] = 2
export_network.columns = ["Source", "Target", "Interaction"]
export_network.to_csv("naive_results/naive_final_network_signed.csv", index=False)

## Tumour-bearing exports

In [ ]:
#tf11=pd.read_csv('/Users/alexren/desktop/Neutrophil/cancer_expression_data.csv')
tf11=pd.read_csv('cancer_results/cancer_expression_data.csv')
#genes = pd.read_csv('/Users/alexren/desktop/Neutrophil/cancer_genes_expressed.csv')
genes = pd.read_csv('cancer_results/cancer_genes_expressed.csv')
genes = genes['x'].to_list()
tf11 = tf11.transpose()
tf11 = tf11[1:100000]
tf11.index = genes

In [ ]:
#restore the average pseudotime values to before log-ing
tf12expression = tf11.apply(lambda x: (math.e**x - 1))
tf12expression[tf12expression <= 0.001] = 0.001
#scale expression
tf12_scaled = tf12expression.div(tf12expression.max(axis=1),axis=0)
tf12expression_logtarget = tf12_scaled.apply(lambda x: np.log2(x))
tf12expression_logtarget = tf12expression_logtarget.dropna()
tf12expression = tf12expression.dropna()

In [ ]:
network0 = pd.read_csv('cancer_results/cancer_initial_network.csv')
network1 = network0[["Source", "Target", "Interaction"]]
gene_list = list(pd.unique(network1['Target']))
network1.columns = ['Source', 'Target', 'Interaction']
ids = list(range(len(gene_list)))

In [ ]:
ob_newall = joblib.load("cancer_results/cancer_networks/n_0.5_k_1.5/cancer_ob_newall_signed.joblib")
res_final = joblib.load("cancer_results/cancer_networks/n_0.5_k_1.5/cancer_res_final_signed.joblib")
combined_network = NetDesDuo.save_combined_network(gene_list, ob_newall, res_final, output="cancer_results/cancer_combined_signed.csv")
export_network = pd.read_csv("cancer_results/cancer_combined_signed.csv")[["Source", "Target", "Interaction"]]
for j in range(len(export_network["Interaction"])):
    if export_network["Interaction"][j] > 1:
        export_network["Interaction"][j] = 1
    else:
        export_network["Interaction"][j] = 2
export_network.columns = ["Source", "Target", "Interaction"]
export_network.to_csv("cancer_results/cancer_final_network_signed.csv", index=False)